<a href="https://colab.research.google.com/github/rezzz59/Hands-On-Large-Language-Models/blob/main/LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=False,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=False,
    device_map="cuda",
    dtype=torch.float16,
    attn_implementation="eager",
)
model.eval()

print("Model siap di:", model.device)

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=False,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=False,
    device_map="cuda",
    dtype=torch.float16,
    attn_implementation="eager",
)
model.eval()

print("Model siap di:", model.device)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model siap di: cuda:0


In [2]:
from transformers import pipeline
# Create a pipeline
generator = pipeline(
 "text-generation",
 model=model,
 tokenizer=tokenizer,
 return_full_text=False,
 max_new_tokens=500,
 do_sample=False
)

In [3]:
# The prompt (user input / query)
messages = [
 {"role": "user", "content": "Create a funny joke about chickens."}
]
# Generate output
output = generator(messages)
print(output[0]["generated_text"])

Why did the chicken join the band? Because it had the drumsticks!


In [4]:
prompt = """Write an email apologizing to Sarah for the tragic gardening mishap.
Explain how it happened.<|assistant|>"""
# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
# Generate the text
generation_output = model.generate(
 input_ids=input_ids,
 max_new_tokens=10
)
# Print the output
print(tokenizer.decode(generation_output[0]))

Write an email apologizing to Sarah for the tragic gardening mishap.
Explain how it happened.<|assistant|> Subject: Sincere Apologies for the


In [5]:
for id in input_ids[0]:
 print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.


Exp
lain
how
it
happened
.
<|assistant|>


In [6]:
teks = "Write an email apologizing for the tragic gardening mishap."
ids = tokenizer.encode(teks)
tokens = tokenizer.convert_ids_to_tokens(ids)

for token_id, token in zip(ids, tokens):
    print(token_id, repr(token))

14350 '▁Write'
385 '▁an'
4876 '▁email'
27746 '▁apolog'
5281 'izing'
363 '▁for'
278 '▁the'
25305 '▁trag'
293 'ic'
16423 '▁garden'
292 'ing'
286 '▁m'
728 'ish'
481 'ap'
29889 '.'


In [7]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

Sub
ject
Subject
:


In [8]:
colors_list = [
 '102;194;165', '252;141;98', '141;160;203',
 '231;138;195', '166;216;84', '255;217;47'
]
def show_tokens(sentence, tokenizer_name):
 tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
 token_ids = tokenizer(sentence).input_ids
 for idx, t in enumerate(token_ids):
    print(
    f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
    tokenizer.decode(t) +
    '\x1b[0m',
    end=' '
    )

In [9]:
# Memanggil fungsi show_tokens yang telah dibuat
show_tokens("Write an email apologizing for the tragic gardening mishap.", model_id)

Write an email apolog izing for the trag ic garden ing m ish ap . 

In [10]:
from transformers import AutoTokenizer

pertanyaan = "Apa arti simbol 🎵?"
bert_tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-cased")

# Jalur 1: emoji dipertahankan
teks_phi3 = pertanyaan

# Jalur 2: teks yang sama diproses dulu oleh tokenizer BERT
bert_ids = bert_tokenizer.encode(
    pertanyaan,
    add_special_tokens=False,
)
teks_bert = bert_tokenizer.decode(
    bert_ids,
    skip_special_tokens=False,
)

print("Pertanyaan awal:      ", pertanyaan)
print("Token BERT:           ", bert_tokenizer.convert_ids_to_tokens(bert_ids))
print("Setelah lewat BERT:   ", teks_bert)

for nama, teks in [
    ("Emoji dipertahankan", teks_phi3),
    ("Setelah tokenizer BERT", teks_bert),
]:
    hasil = generator([{"role": "user", "content": teks}])
    print(f"\n{nama}")
    print("Input ke model:", teks)
    print("Jawaban model:", hasil[0]["generated_text"])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Pertanyaan awal:       Apa arti simbol 🎵?
Token BERT:            ['A', '##pa', 'art', '##i', 'si', '##mbo', '##l', '[UNK]', '?']
Setelah lewat BERT:    Apa arti simbol [UNK]?

Emoji dipertahankan
Input ke model: Apa arti simbol 🎵?
Jawaban model: 🎵 adalah simbol yang sering digunakan untuk menunjukkan musik atau lagu. Dalam bahasa Indonesia, 🎵 dikenal sebagai "Gambar Lagu" atau "Gambar Musik". Ini sering digunakan dalam media sosial, pesan, atau komentar untuk menunjukkan kesuksesan atau kegembiraan dalam musik atau lagu yang dijual atau diproduksi.

Setelah tokenizer BERT
Input ke model: Apa arti simbol [UNK]?
Jawaban model: Simbol [UNK] biasanya digunakan dalam konteks komputer, pengolahan teks, atau pengolahan data untuk menunjukkan adanya suatu karakter atau teks yang tidak ditemukan atau tidak tersedia. Ini bisa berarti suatu karakter dalam bahasa tertentu yang tidak tersedia dalam dataset atau karakter yang tidak dapat dibaca karena kesalahan pemetaan.

Untuk memahami apa yang dim

In [11]:
print("Token BERT:", bert_tokenizer.convert_ids_to_tokens(bert_ids))
print("Teks hasil decode:", repr(teks_bert))
assert bert_tokenizer.unk_token_id in bert_ids

Token BERT: ['A', '##pa', 'art', '##i', 'si', '##mbo', '##l', '[UNK]', '?']
Teks hasil decode: 'Apa arti simbol [UNK]?'


**# EMBEDDING**

In [12]:
from transformers import AutoModel, AutoTokenizer
# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")
# Load a language model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")
# Tokenize the sentence
tokens = tokenizer('Hello world', return_tensors='pt')
# Process the tokens
output = model(**tokens)[0]

config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


... (2 outputs left out)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d

model.safetensors: reconstructing file:   0%|          |  0.00B /  241MB            

model.safetensors: downloading bytes:           |  0.00B            

In [13]:
tokens = tokenizer('Hello world ahmad', return_tensors='pt')
# Process the tokens
output = model(**tokens)[0]

In [14]:
output.shape

torch.Size([1, 6, 384])

In [15]:
for token in tokens['input_ids'][0]:
 print(tokenizer.decode(token))

[CLS]
Hello
 world
 ah
mad
[SEP]


In [16]:
tokens['input_ids'][0]

tensor([    1, 31414,   232, 23184, 17497,     2])

In [17]:
tokenizer.decode(31414), tokenizer.decode(232)

('Hello', ' world')

In [18]:
from sentence_transformers import SentenceTransformer
# Load model
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
# Convert text to text embeddings
vector = model.encode("Best movie ever!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [19]:
vector.shape

(768,)

##word embeeding

In [20]:
!pip install gensim
import gensim.downloader as api
# Download embeddings (66MB, glove, trained on wikipedia, vector size: 50)
# Other options include "word2vec-google-news-300"
# More options at https://github.com/RaRe-Technologies/gensim-data
model = api.load("glove-wiki-gigaword-50")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 29.6 MB/s  0:00:00
[==================================================] 100.0% 66.0/66.0MB downloaded


In [21]:
model.most_similar([model['king']], topn=11)

[('king', 1.0000001192092896),
 ('prince', 0.8236179351806641),
 ('queen', 0.7839043140411377),
 ('ii', 0.7746230363845825),
 ('emperor', 0.7736247777938843),
 ('son', 0.766719400882721),
 ('uncle', 0.7627150416374207),
 ('kingdom', 0.7542161345481873),
 ('throne', 0.7539914846420288),
 ('brother', 0.7492411136627197),
 ('ruler', 0.7434253692626953)]

In [22]:
import pandas as pd
from urllib import request
# Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')
# Parse the playlist dataset file. Skip the first two lines as
# they only contain metadata
lines = data.read().decode("utf-8").split('\n')[2:]
# Remove playlists with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]
# Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

In [23]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

In [24]:
from gensim.models import Word2Vec
# Train our Word2Vec model
model = Word2Vec(
 playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4
)

In [25]:
song_id = 6728
# Ask the model for songs similar to song #2172
model.wv.most_similar(positive=str(song_id))

[('15385', 0.9973079562187195),
 ('28211', 0.9972243905067444),
 ('6722', 0.9969748258590698),
 ('19373', 0.996900200843811),
 ('65296', 0.9967913627624512),
 ('31522', 0.9967448711395264),
 ('30848', 0.9966646432876587),
 ('21714', 0.9966468811035156),
 ('19530', 0.9966256022453308),
 ('5920', 0.9966216683387756)]

In [26]:
print(songs_df.iloc[6728])

title     Sitting Home
artist           Total
Name: 6728 , dtype: object


In [27]:
import numpy as np
def print_recommendations(song_id):
 similar_songs = np.array(
 model.wv.most_similar(positive=str(song_id),topn=5)
 )[:,0]
 return songs_df.iloc[similar_songs]
# Extract recommendations
print_recommendations(song_id)

,title,artist
id,,
15385,In The Mood,The Whispers
28211,Can't Live Without You,Charlie Wilson
6722,Don't Walk Away,Jade
19373,Anywhere You Are,Pieces Of A Dream
65296,The Answer,Raphael Saadiq


#SECTION 3

In [1]:
!pip install --upgrade transformers accelerate
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3-mini-4k-instruct",
 device_map="cuda",
 torch_dtype="auto",
 trust_remote_code=False,
)
# Create a pipeline
generator = pipeline(
 "text-generation",
 model=model,
 tokenizer=tokenizer,
 return_full_text=False,
 max_new_tokens=50,
 do_sample=False,
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 79.5 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.1
    Uninstalling transformers-5.22.1:
      Successfully uninstalled transformers-5.22.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.15.0
    Uninstalling accelerate-1.15.0:
      Successfully uninstalled accelerate-1.15.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [accelerate]


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

... (3 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [2]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap.Explain how it happened."
output = generator(prompt)
print(output[0]['generated_text'])

Express your sincere apologies.Offer to make amends.


Dear Sarah,


I hope this message finds you well. I am writing to you with a heavy heart to apologize for the unfortunate


In [3]:
# Menampilkan arsitektur model Phi-3 yang sudah di-load sebelumnya
print(model)

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur

sampling/decoding

In [4]:

prompt = "The capital of Kalimantan Selatan is"
# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
# Tokenize the input prompt
input_ids = input_ids.to("cuda")
# Get the output of the model before the lm_head
model_output = model.model(input_ids)
# Get the output of the lm_head
lm_head_output = model.lm_head(model_output[0])

In [5]:
token_id = lm_head_output[0,-1].argmax(-1)
tokenizer.decode(token_id)

'Ban'

In [7]:
prompt = "Write a very long email apologizing to Sarah for the tragic gardeningmishap. Explain how it happened."
# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
input_ids = input_ids.to("cuda")

In [8]:
%%timeit -n 1
# Generate the text
generation_output = model.generate(
 input_ids=input_ids,
 max_new_tokens=100,
 use_cache=True
)

5.53 s ± 249 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [9]:
%%timeit -n 1
# Generate the text
generation_output = model.generate(
 input_ids=input_ids,
 max_new_tokens=100,
 use_cache=False
)

33.1 s ± 822 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
